# Pipeline Cálculo Vectorial — Exploración 2026-1

Notebook principal para procesar notas del ciclo 2026-1.  
Ejecutar las celdas en orden.

**Flujo de datos:**
```
Gradescope (corrección)
    ↓  Post Grades to Canvas  ← el docente hace click una vez
Canvas Gradebook (Tareas, APs, EAs, RCs, Exámenes)
    ↓  fetch_canvas_grades_grupo()  ← el pipeline descarga todo
Google Sheets Dashboard
```

**Orden de ejecución:**
1. Imports y configuración
2. Carga del dashboard base (Notas)
3. Descarga de notas desde Canvas API
4. Merge de todas las fuentes
5. Ejecución de cálculos
6. Visualización de resultados
7. Exportar CSV de salida
8. (Opcional) Actualizar Google Sheets

## 1. Imports y configuración de paths

In [ ]:
import sys
import os
import yaml
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Agregar el directorio raíz al path para importar src/
RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# Cargar variables de entorno (.env → CANVAS_TOKEN, CANVAS_BASE_URL)
load_dotenv(RAIZ / ".env")

from src.canvas_api import fetch_canvas_grades_grupo, CANVAS_A_PIPELINE
from src.merge import merge_con_fallback, merge_todas_fuentes, reporte_merge
from src.calculos import ejecutar_calculos
from src.reporte import (
    estadisticas_por_seccion,
    alumnos_en_riesgo,
    resumen_curso,
)

# Configuración de paths
CICLO = "2026-1"
DIR_DATA = RAIZ / "data" / CICLO
DIR_PROCESADO = DIR_DATA / "processed"
DIR_OUTPUT = DIR_DATA / "output"

# Cargar configuración del ciclo
with open(RAIZ / "config" / f"{CICLO}.yaml") as f:
    CONFIG = yaml.safe_load(f)

print(f"Ciclo: {CONFIG['ciclo']}")
print(f"Directorio de datos: {DIR_DATA}")
print(f"Secciones auditorio: {CONFIG['secciones']['auditorio']}")
print(f"Secciones aula: {CONFIG['secciones']['aula']}")
print(f"CANVAS_TOKEN cargado: {'✅' if os.getenv('CANVAS_TOKEN') else '❌ — revisar .env'}")

## 2. Carga del dashboard base (Notas)

Lista oficial de alumnos matriculados, descargada desde el Google Sheet 'Notas'.

In [ ]:
ARCHIVO_BASE = DIR_DATA / "raw" / "Notas.csv"

df_base = pd.read_csv(
    ARCHIVO_BASE,
    sep=",",
    encoding="utf-8",
    dtype=str,
)

# Limpiar filas completamente vacías
df_base = df_base.dropna(how="all").reset_index(drop=True)

if "Código" in df_base.columns:
    df_base["Código"] = df_base["Código"].str.strip()
if "Correo" in df_base.columns:
    df_base["Correo"] = df_base["Correo"].str.strip()

print(f"Dashboard base cargado: {len(df_base)} alumnos")
print(f"Columnas: {list(df_base.columns)}")
df_base.head()

## 3. Descarga de notas desde Canvas API

Todo viene de Canvas: Tareas, Actividades Previas, EAs, RCs.  
Las EAs se corrigen en Gradescope y el docente hace **Post Grades to Canvas** una vez por evaluación.

- **Auditorios** (secciones 1, 2): Tareas T1–T6 + APs
- **Aulas** (secciones 11–24): EAs 1–6 + RC1, RC2

In [ ]:
# --- 3.1 Auditorios: Tareas T1–T6 + Actividades Previas ---
courses_aud = CONFIG["canvas_api"]["courses_auditorio"]
assignments_aud = CONFIG["canvas_api"]["assignments_auditorio"]

print("Descargando notas de auditorios...")
df_canvas_aud = fetch_canvas_grades_grupo(
    courses=courses_aud,
    assignment_names=assignments_aud,
    nombre_a_col=CANVAS_A_PIPELINE,
)
print(f"✅ Auditorios: {len(df_canvas_aud)} alumnos")
print(f"   Columnas descargadas: {[c for c in df_canvas_aud.columns if c not in ['Código','Correo','Sección']]}")
df_canvas_aud.head()

In [ ]:
# --- 3.2 Aulas: EAs 1–6 + RC1, RC2 ---
courses_aula = CONFIG["canvas_api"]["courses_aula"]
assignments_aula = CONFIG["canvas_api"]["assignments_aula"]

print("Descargando notas de aulas...")
df_canvas_aula = fetch_canvas_grades_grupo(
    courses=courses_aula,
    assignment_names=assignments_aula,
    nombre_a_col=CANVAS_A_PIPELINE,
)
print(f"✅ Aulas: {len(df_canvas_aula)} alumnos")
print(f"   Columnas descargadas: {[c for c in df_canvas_aula.columns if c not in ['Código','Correo','Sección']]}")
df_canvas_aula.head()

In [ ]:
# --- 3.3 Consolidar: un único DataFrame con todos los alumnos ---
df_canvas_all = pd.concat([df_canvas_aud, df_canvas_aula], ignore_index=True)
df_canvas_all = df_canvas_all.drop_duplicates(subset=["Código"], keep="first")

print(f"Canvas consolidado: {len(df_canvas_all)} alumnos")
print(f"Columnas totales: {list(df_canvas_all.columns)}")
df_canvas_all.head()

## 4. Merge de todas las fuentes

Llave primaria: `Código`. Fallback: `Correo` para alumnos con código especial.

In [ ]:
# Columnas de notas disponibles en Canvas (todo excepto las llaves de identificación)
cols_canvas = [c for c in df_canvas_all.columns if c not in ["Código", "Correo", "Sección"]]

fuentes = {}

if not df_canvas_all.empty:
    fuentes["canvas"] = {
        "df": df_canvas_all,
        "columnas": cols_canvas + ["Sección"],
        "sufijo": "_canvas",
    }

# Ejecutar todos los merges
df_merged = merge_todas_fuentes(df_base, fuentes, verbose=True)

# Reporte de cobertura
reporte = reporte_merge(df_merged)
df_merged.shape

## 5. Ejecución de cálculos

Calcula PT1, PT2, PEA1, PEA2, BPEA1_prov, BPEA2, PfEA1, PfEA2, TA1, TA2, EP, EF, NF.

In [ ]:
COLS_AP_BPEA1 = [
    "AP1V1", "AP1V2",
    "AP2V1", "AP2V2",
    "AP3V1", "AP3V2",
    "AP4V1", "AP4V2",
    "AP5V1", "AP5V2",
    "AP6V1", "AP6V2",
]

# Semana 9 tiene 2 videos (AP7V1, AP7V2), resto 1 video cada una
COLS_AP_BPEA2 = ["AP7V1", "AP7V2", "AP8V1", "AP9V1", "AP10V1"]

df_calculado = ejecutar_calculos(
    df_merged,
    CONFIG,
    cols_ap_bpea1=COLS_AP_BPEA1,
    cols_ap_bpea2=COLS_AP_BPEA2,
)

print(f"DataFrame calculado: {df_calculado.shape}")

## 6. Visualización de resultados por sección

In [ ]:
# Estadísticas globales del curso
df_resumen = resumen_curso(df_calculado)

In [ ]:
# Estadísticas por sección para NF (o cualquier columna calculada)
if "NF" in df_calculado.columns:
    df_stats_seccion = estadisticas_por_seccion(df_calculado, "NF")
    print("\nEstadísticas de NF por sección:")
    display(df_stats_seccion)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Boxplot por sección
    df_plot = df_calculado[["Sección", "NF"]].dropna()
    if not df_plot.empty:
        df_plot["Sección"] = df_plot["Sección"].astype(str)
        sns.boxplot(data=df_plot, x="Sección", y="NF", ax=axes[0], palette="muted")
        axes[0].set_title("Distribución de NF por Sección")
        axes[0].set_ylabel("Nota Final")
        axes[0].axhline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
        axes[0].legend()

    # Histograma de NF global
    df_calculado["NF"].dropna().hist(
        bins=20, ax=axes[1], color="steelblue", edgecolor="white"
    )
    axes[1].axvline(10.5, color="red", linestyle="--", label="Mín. aprobatorio")
    axes[1].set_title("Distribución Global de NF")
    axes[1].set_xlabel("Nota Final")
    axes[1].set_ylabel("Número de alumnos")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(DIR_OUTPUT / "distribucion_NF.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Alumnos en riesgo (NF proyectada < 10.5)
df_riesgo = alumnos_en_riesgo(df_calculado, umbral=10.5)
if not df_riesgo.empty:
    display(df_riesgo)

## 7. Exportar CSV de salida

Guarda el DataFrame final como respaldo local en `data/2026-1/output/`.

In [ ]:
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

ARCHIVO_SALIDA = DIR_OUTPUT / "notas_calculadas_2026-1.csv"
df_calculado.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8")
print(f"✅ Notas exportadas: {ARCHIVO_SALIDA}")
print(f"   Filas: {len(df_calculado)}, Columnas: {len(df_calculado.columns)}")

## 8. (Opcional) Actualizar Google Sheets

⚙️ Requiere:
- Archivo `credentials/service_account.json`
- Sheet ID configurado en `config/2026-1.yaml`
- Sheet compartido con el email de la Service Account

In [ ]:
# Descomentar para ejecutar la actualización de Google Sheets

# from src.gdrive import conectar_gspread, escribir_columnas_notas, escribir_formativa
#
# CREDENTIALS = RAIZ / "credentials" / "service_account.json"
# SHEET_ID = CONFIG["dashboard"]["sheet_id"]
#
# gc = conectar_gspread(CREDENTIALS)
#
# columnas_a_escribir = [
#     "EA1", "EA2", "EA3",
#     "PT1", "PEA1", "BPEA1_prov", "PfEA1",
#     "TA1", "EP", "EF", "NF",
# ]
#
# escribir_columnas_notas(
#     sheet_id=SHEET_ID,
#     df=df_calculado,
#     columnas=columnas_a_escribir,
#     gc=gc,
# )
#
# escribir_formativa(
#     sheet_id=SHEET_ID,
#     df_formativa=df_calculado,
#     gc=gc,
# )
print("(Celda de actualización de Google Sheets — descomentar para usar)")